In [1]:
# here we continue inst tuning by tuning a GPTLarge model to be a chatbot

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoModelForCausalLM, GPT2Tokenizer
from datasets import load_dataset

import textwrap

In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

Lenght of questions and answers

In [3]:
# load pretrainned GPT2L model and tokenizer
gpt2  = AutoModelForCausalLM.from_pretrained('gpt2-large')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2') #all GPT variants use the same tokenizer
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

In [4]:
gpt2
# ndim here is 1280 as compared to 768 in gpt-small
# 36 transformer blocks


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [5]:
dataset = load_dataset('THUDM/webglm-qa')
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'references'],
        num_rows: 43579
    })
    validation: Dataset({
        features: ['question', 'answer', 'references'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'answer', 'references'],
        num_rows: 400
    })
})

In [6]:
# loop through samples and get the token length of questions and answers
sampleLengths = np.zeros((len(dataset['train']),2))
for i in range(sampleLengths.shape[0]):
    sampleLengths[i,0] = len(tokenizer.encode(dataset['train'][i]['question']))
    sampleLengths[i,1] = len(tokenizer.encode(dataset['train'][i]['answer']))

![title](../images/length_QnA.png)

In [8]:
# QUestions are short compared to answers
#    will that be problematic, like will it introduce a bias in training???
#        --- Yes, model is biased that questions will be shorter and answers will be longner
#        --- But this bias is not problematic, its a natural feature of QnA queires work

Creating question-starting batches

In [9]:
# Here we each training sample will start with a question, this is diff to previous excercise where it was random
# selection from a very long vector of [questions, answers, question, answers, question.....]

# HEre we always start the sample with a question
# also every question and answer have varyung lengths , so we need to pad shorter seq and truncate longer seq,
     # so that every row in the matrix has seq_len no of cols
# since its padded, we need to use associated attention mask

In [10]:
seq_len = 256

In [11]:
#initialising (just using the first 10k data samples)
trainTokens = torch.full((10000,seq_len), tokenizer.pad_token_id) #creates a 2D matrix init with pad_tokens
testTokens = torch.full((1000,seq_len), tokenizer.pad_token_id)

# loop over tokens
for idx in range(trainTokens.shape[0]):
    # construct the token seq
    txt = f"QUESTION: {dataset['train'][idx]['question']} ANSWER: {dataset['train'][idx]['answer']}"
    tokz  = tokenizer.encode(txt, add_special_tokens=True)

    # insert this seq to data matrix, truncating when needed
    endOfSeq = min(seq_len, len(tokz))
    trainTokens[idx,:endOfSeq] = torch.tensor(tokz[:endOfSeq])

#repeat for test tokens

for idx in range(testTokens.shape[0]):
    # construct the token seq
    txt = f"QUESTION: {dataset['validation'][idx]['question']} ANSWER: {dataset['validation'][idx]['answer']}"
    tokz  = tokenizer.encode(txt, add_special_tokens=True)
    endOfSeq = min(seq_len, len(tokz))
    testTokens[idx,:endOfSeq] = torch.tensor(tokz[:endOfSeq])

In [12]:
# attn_mask for 1 row
attn_mask = (trainTokens[0] != tokenizer.pad_token_id).long()

print(f'Training tokens:\n{trainTokens[0]}\n')
print(f'Attention mask:\n{attn_mask}')

Training tokens:
tensor([35780,  2849,    25,   287,  4346, 45038,   262,   966,   286, 24430,
          262,   717,   734,  5341,   351,   257, 10484,   532,   510,   262,
         3504,   532,   407,  3218, 10484,  5341,  1312,   651,   883,  3537,
        17887,  1137,    25,   383,   966,   286, 24430,   262,   717,   734,
         5341,   351,   257, 10484,   510,   262,  3504,   318,   284,  1011,
         4621,   286,   262,  2319,  1218,   711,  8801,   290,   262,   734,
         5664,  6509,   287,  4708,   290,  4152,  4346,  1830,    13,  2750,
         2491,   262,  2613,  3264,   510,   262,  3504,    11,   262,  6907,
         7176,   284,   307,  1498,   284,  2512,  2506,   510,   290,   262,
         2491,   736,   460,   787,   257,  1048,  2051,   290,   886,   510,
          287,   262,   886,  6516,    58,    17,  4083, 18162,   510,   262,
         3504,   318,   635,   262, 35581,  3108,   284,   262,   886,  6516,
           11,   523,   340,   318,  1690,  356

In [13]:
# check a random batch
# check a rabdom batcgh
ix = np.random.randint(0, trainTokens[0].shape,8)
X = trainTokens[ix]
attn_mask = (X != tokenizer.pad_token_id).long()


print(f'Size ofo batch: {X.shape}')
print(f'Size of attn mask: {attn_mask.shape}\n')
print('some examples:')
for t in range(5):
    print(f'*** Example: \n', textwrap.fill(tokenizer.decode(X[t]),123),'\n')

Size ofo batch: torch.Size([8, 256])
Size of attn mask: torch.Size([8, 256])

some examples:
*** Example: 
 QUESTION: We all mostly skip or block ads. What makes companies still believe online ads like on youtube is worth
investing? ANSWER: Companies still believe that investing in online ads, such as on YouTube, is worth it because they offer
valuable, native ads that feel like a part of the overall YouTube experience, and that won't frustrate viewers and drive
them to turn on AdBlock[1]. Additionally, they can serve ads pre-roll or mid-roll that viewers can’t skip, and pay per
impression at a cost per 1,000 views[2]. Furthermore, even though people have the ability to block YouTube ads, the
majority of YouTube’s 2.56 billion users don’t have an ad blocker installed, meaning companies are still likely to reach
their target audience[4]. Finally, YouTube accepts the loss of ad costs from ad blockers because their main goal is to have
as many people using their platform as possible[5].<|

In [14]:
X

tensor([[35780,  2849,    25,  ..., 50256, 50256, 50256],
        [35780,  2849,    25,  ..., 50256, 50256, 50256],
        [35780,  2849,    25,  ..., 50256, 50256, 50256],
        ...,
        [35780,  2849,    25,  ..., 50256, 50256, 50256],
        [35780,  2849,    25,  ..., 50256, 50256, 50256],
        [35780,  2849,    25,  ..., 50256, 50256, 50256]])

In [15]:
# count the % of pad tokens in train and test set
aveAM = (trainTokens == tokenizer.pad_token_id).sum()/torch.numel(trainTokens)
print(f'{aveAM*100:5.2f}% of TRAIN token positions are EOS')

aveAM = (testTokens == tokenizer.pad_token_id).sum()/torch.numel(testTokens)
print(f'{aveAM*100:5.2f}% of TEST token positions are EOS')

38.86% of TRAIN token positions are EOS
38.57% of TEST token positions are EOS


In [17]:
# almost 40% of tokens are just padded tokens which are not used in downstream caluclations


#SO MODEL DOESNT BENEFIT FROM HAVING iN THIS FORMAT (always strating with QUESTION, followed by an answer)
   # because the model learns that pattern ---> that somewhere there is a question which are 10ish long tokens and ana answeer which are longer
   # it doesnt matter where the pattern starts and ends inside the batch

FINE TUNE THE MODEL

In [18]:
gpt2 = gpt2.to(device)
optimizer = torch.optim.AdamW(gpt2.parameters(), lr=5e-5, weight_decay=.01)

In [19]:
num_samples = 123
batch_size = 8
#init the loss
train_loss = np.zeros(num_samples)
test_loss = np.zeros(num_samples)

for sampli in range(num_samples):
    ix = np.random.randint(0, trainTokens[0].shape,batch_size)
    X = trainTokens[ix]
    attn_mask = (X != tokenizer.pad_token_id).long()

    # move data to GPU
    attn_mask = attn_mask.to(device)
    X = X.to(device)

    
    #fwd pass
    gpt2.zero_grad()
    outputs= gpt2(X,labels=X, attention_mask = attn_mask)
    loss = outputs.loss

    #backprop
    loss.backward()
    optimizer.step()
    
    # sum the batch loss
    train_loss[sampli] = loss.item()
    if sampli%5==0:
        # get a btch of dat and create a mask
        ix = np.random.randint(0, testTokens[0].shape,batch_size)
        X = testTokens[ix]
        attn_mask = (X != tokenizer.pad_token_id).long()
        
        attn_mask = attn_mask.to(device)
        X = X.to(device)

        with torch.no_grad():
            gpt2.eval()
            outputs = gpt2(X,labels=X, attention_mask = attn_mask)
            test_loss[sampli]  = outputs.loss.item()
        gpt2.train()
        print(f'Sample {sampli}/{num_samples}, train/test loss: {train_loss[sampli]} / {test_loss[sampli]}')

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Sample 0/123, train/test loss: 6.767118453979492 / nan


RuntimeError: MPS backend out of memory (MPS allocated: 26.90 GiB, other allocations: 302.33 MiB, max allowed: 27.20 GiB). Tried to allocate 10.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [10]:
# Qualitative assesment
# prompt = 'QUESTION: Would it be strange to have a pet rock and feed it styrofoam?'
# prompt = 'QUESTION: What is the meaning of "obsolete"'
prompt = 'QUESTION: Where did the word "butterfly" come from?'
in2gpt = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

output = gpt2.generate(in2gpt, max_length = 100, pad_token_id=50256, do_sample=True)
print(tokenizer.decode(in2gpt[0].cpu()),'\n')
print(textwrap.fill(tokenizer.decode(output[0][len(in2gpt[0]):]),60)) # decode only the actual LLM answer part

# this is stochastic sampling, so we will get diff results everytime we run the same prompt

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/Users/raeez/.pyenv/versions/jupyter-env/lib/python3.10/site-packages/transformers/generation/utils.py:2636: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on mps, whereas the model is on cpu. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cpu') before running `.generate()`.
  warnings.warn(


RuntimeError: Placeholder storage has not been allocated on MPS device!